# Lab 2 - Modelos Tradicionais

Classificadores: **KNN**, **Árvore de Decisão** e **SVM**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import os
from datetime import datetime

## Carregamento dos Dados Processados

In [ ]:
data_dir = './dados_processados'

X_train = pd.read_parquet(f'{data_dir}/X_train.parquet')
y_train = pd.read_parquet(f'{data_dir}/y_train.parquet').values.ravel()
X_test = pd.read_parquet(f'{data_dir}/X_test.parquet')
y_test = pd.read_parquet(f'{data_dir}/y_test.parquet').values.ravel()

mod_time = datetime.fromtimestamp(os.path.getmtime(f'{data_dir}/X_train.parquet'))
print(f'Dados carregados (gerados em {mod_time.strftime("%Y-%m-%d %H:%M")})')
print(f'  X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'  y_train: {y_train.shape}  |  y_test: {y_test.shape}')
print(f'  Features: {list(X_train.columns)}')
print(f'  Churn rate treino: {y_train.mean():.3f}  |  teste: {y_test.mean():.3f}')

## Configuração do MLflow e Função de Avaliação

O MLflow registra automaticamente parâmetros, métricas e artefatos de cada modelo.
Todos os notebooks usam o mesmo experimento (`Lab2-Churn`), então os runs aparecem juntos na UI.

**Para visualizar os resultados após rodar os notebooks:**
```bash
cd notebooks
mlflow ui
# Abrir http://127.0.0.1:5000 no navegador
```

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

# Mesmo experimento em todos os notebooks — runs aparecem juntos na UI
mlflow.set_experiment("Lab2-Churn")


def avaliar_modelo(nome, y_true, y_pred):
    """Avalia um modelo, printa o relatório e retorna dict com métricas."""
    metricas = {
        'Modelo': nome,
        'Acuracia': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred)
    }
    print(f'\n=== {nome} ===')
    print(classification_report(y_true, y_pred, target_names=['Não cancelou', 'Cancelou']))
    print(f'Kappa: {metricas["Kappa"]:.4f}')
    return metricas


def logar_mlflow(metricas, params, artefatos=None):
    """Registra métricas, parâmetros e artefatos no run MLflow ativo."""
    for k, v in params.items():
        mlflow.log_param(k, v)
    for k, v in metricas.items():
        if k != 'Modelo':
            mlflow.log_metric(k.lower().replace('-', '_'), v)
    if artefatos:
        for caminho in artefatos:
            mlflow.log_artifact(caminho)

## Normalização

KNN e SVM são sensíveis à escala das features — aplicamos `StandardScaler`.
Árvore de Decisão **não precisa** de normalização e usa `X_train`/`X_test` diretamente.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Dados normalizados — média ~0, std ~1')
print(f'  X_train_scaled: {X_train_scaled.shape}')
print(f'  X_test_scaled:  {X_test_scaled.shape}')

## Exemplo de Uso — Padrão MLflow

O bloco abaixo mostra o padrão que deve ser repetido para cada modelo.
Cada `with mlflow.start_run(...)` cria um **run** separado no experimento.

```
with mlflow.start_run(run_name="NomeDoModelo"):
    1. Definir e logar parâmetros
    2. Treinar o modelo
    3. Predizer e avaliar
    4. Logar métricas e artefatos
    5. (Opcional) Salvar o modelo treinado
```

In [ ]:
# ─── Exemplo: KNN ───────────────────────────────────────────────
from sklearn.neighbors import KNeighborsClassifier

with mlflow.start_run(run_name="KNN"):
    # 1. Parâmetros
    k = 5
    params = {'modelo': 'KNN', 'n_neighbors': k, 'scaler': 'StandardScaler'}

    # 2. Treinar (com dados normalizados — KNN é sensível à escala)
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)

    # 3. Predizer e avaliar
    y_pred = model.predict(X_test_scaled)
    metricas = avaliar_modelo('KNN', y_test, y_pred)

    # 4. Logar no MLflow
    logar_mlflow(metricas, params)